In [11]:
from transformers import pipeline
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
from datetime import datetime

# Load sentiment model
model_name = "cardiffnlp/twitter-roberta-base-sentiment"
classifier = pipeline("sentiment-analysis", model=model_name)

# Label mapping
label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Neutral",
    "LABEL_2": "Positive"
}

results_list = []

# Function to process uploaded CSV with optional timestamp column
def analyze_csv(file):
    global results_list
    df = pd.read_csv(file.name)

    if "text" in df.columns and "date" in df.columns:
        tweets = df["text"].dropna().tolist()
        dates = pd.to_datetime(df["date"].dropna().tolist())
    else:
        tweets = df.iloc[:, 0].dropna().tolist()
        dates = [None] * len(tweets)

    results_list = []
    output_text = ""

    for tweet, date in zip(tweets[:100], dates[:100]):  # Limit to 100
        result = classifier(tweet)[0]
        sentiment = label_map[result["label"]]
        confidence = round(result["score"], 3)
        date_str = date.strftime("%Y-%m-%d") if date is not None else "N/A"
        results_list.append((tweet, sentiment, confidence, date_str))
        output_text += f"{date_str} - {sentiment} ({confidence}): {tweet}\n\n"

    return output_text, None, None  # Clear plots initially

# Bar chart display
def show_chart():
    if not results_list:
        return None
    df = pd.DataFrame(results_list, columns=["Tweet", "Sentiment", "Confidence", "Date"])
    counts = df["Sentiment"].value_counts()

    fig, ax = plt.subplots()
    counts.plot(kind='bar', color=['red', 'gray', 'green'], ax=ax)
    plt.title("Sentiment Distribution")
    plt.ylabel("Count")
    plt.xticks(rotation=0)
    plt.grid(True)
    plt.tight_layout()
    return fig

# Time series line chart
def show_time_series():
    if not results_list:
        return None
    df = pd.DataFrame(results_list, columns=["Tweet", "Sentiment", "Confidence", "Date"])
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df.dropna(subset=["Date"], inplace=True)
    if df.empty:
        return None
    time_df = df.groupby(["Date", "Sentiment"]).size().unstack(fill_value=0)

    fig, ax = plt.subplots()
    time_df.plot(ax=ax)
    plt.title("Sentiment Over Time")
    plt.xlabel("Date")
    plt.ylabel("Tweet Count")
    plt.grid(True)
    plt.tight_layout()
    return fig

# Save results function (can be added as button if needed)
def save_results():
    if not results_list:
        return "No results to save!"
    df = pd.DataFrame(results_list, columns=["Tweet", "Sentiment", "Confidence", "Date"])
    df.to_csv("covid_sentiment_results.csv", index=False)
    return "✅ Results saved as covid_sentiment_results.csv"

with gr.Blocks() as demo:
    gr.Markdown("# COVID-19 Tweet Sentiment Analyzer")
    gr.Markdown("Upload a CSV file with tweets (and optional 'date' column) to get sentiment analysis and visualizations.")

    file_input = gr.File(label="Upload CSV (with 'text' and optional 'date' columns)")
    output_text = gr.Textbox(label="Sentiment Analysis Results", lines=15)
    bar_chart = gr.Plot(label="Sentiment Distribution")
    time_series_chart = gr.Plot(label="Sentiment Over Time")

    analyze_btn = gr.Button("Analyze Tweets")
    bar_btn = gr.Button("Show Sentiment Distribution")
    time_btn = gr.Button("Show Sentiment Over Time")

    analyze_btn.click(analyze_csv, inputs=file_input, outputs=[output_text, bar_chart, time_series_chart])
    bar_btn.click(show_chart, inputs=None, outputs=bar_chart)
    time_btn.click(show_time_series, inputs=None, outputs=time_series_chart)

demo.launch()


Device set to use cpu


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ae23ddf64286bb41a6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
